# 🤖 Chatbot Practice (Solution)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nawooooojin/llm-api-hands-on/blob/main/Practice_solution/02-Chatbot-Practice_solution.ipynb)

## 1) 환경 준비

In [ ]:
%pip install -q -U openai ipywidgets

### API 키 설정 및 Client 생성
- **Colab**: 왼쪽 메뉴 `보안 비밀(열쇠 아이콘)`에 `OPENAI_API_KEY`를 저장하고 **노트북 액세스**를 켜 두면 자동으로 불러옵니다.
- **로컬**: 환경 변수 `OPENAI_API_KEY`를 미리 설정해 두거나, 아래 셀을 실행했을 때 뜨는 입력창에 키를 붙여넣으면 됩니다.
- 아래의 코드에서 오류가 난다면 API 키의 오류일 가능성이 높습니다.

In [1]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    try:
        from google.colab import userdata  # type: ignore
        OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")
    except ImportError:
        pass  # Colab이 아닌 환경
    except Exception:
        print("Colab 메뉴의 보안 비밀(열쇠 아이콘)에 OPENAI_API_KEY가 저장되어 있는지, 노트북 액세스가 켜져 있는지 확인해주세요.")

if not OPENAI_API_KEY:
    from getpass import getpass
    OPENAI_API_KEY = getpass("OPENAI_API_KEY를 입력하세요: ")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("✅ API 키 설정 완료")

✅ API 키 설정 완료


In [2]:
from openai import OpenAI

client = OpenAI()
print("✅ 클라이언트 생성 완료")

✅ 클라이언트 생성 완료


### OpenAI API 호출 실습

In [3]:
def create_greeting(client, model, instructions, user_prompt):
    """Responses API 호출 솔루션"""
    response = client.responses.create(
        model=model,
        instructions=instructions,
        input=user_prompt,
    )
    return response.output_text

In [4]:
greeting = create_greeting(
    client=client,
    model="gpt-6-luna",
    instructions="You are a helpful assistant. Reply in Korean.",
    user_prompt="오늘 하루를 기분 좋게 시작할 수 있는 인사말을 한 문장으로 만들어 주세요."
)
print(greeting)

좋은 아침이에요! 오늘도 작은 행복이 가득한 멋진 하루 보내세요.


## 2) 멀티턴 대화 구성 정답

In [5]:
def request_chat_completion(
    client,
    model,
    history,
    *,
    reasoning_effort="medium",
    verbosity="medium"
):
    """Responses API 멀티턴 대화 솔루션 구현."""

    kwargs = {}
    if reasoning_effort:
        kwargs["reasoning"] = {"effort": reasoning_effort}
    if verbosity:
        kwargs["text"] = {"verbosity": verbosity}

    # API 호출
    response = client.responses.create(
        model=model,
        input=history,
        **kwargs,
    )
    return response.output_text, response

In [6]:
conversation = [
    {
        "role": "developer",
        "content": "You are a helpful assistant. Reply in Korean and keep answers concise.",
    },
    {
        "role": "user",
        "content": "방금 읽은 책을 한 줄로 요약해 달라고 부탁하는 방법을 알려 줘.",
    },
]

first_text, first_response = request_chat_completion(
    client=client,
    model="gpt-6-luna",
    history=conversation,
    reasoning_effort="medium",
    verbosity="low"
)
print("첫 번째 응답:", first_text)

conversation.append({"role": "assistant", "content": first_text})
conversation.append({"role": "user", "content": "이 내용을 더 자세하게 바꿔 줄래?"})

second_text, second_response = request_chat_completion(
    client=client,
    model="gpt-6-luna",
    history=conversation,
    verbosity="high"
)
print("\n두 번째 응답:", second_text)

첫 번째 응답: “방금 읽은 《[책 제목]》의 핵심 메시지를 한 문장으로 요약해 줘.”라고 부탁하면 돼요.



두 번째 응답: 이렇게 부탁하면 더 구체적이에요:

“방금 읽은 《[책 제목]》의 핵심 주장과 주요 내용을 바탕으로, 저자가 독자에게 전하려는 메시지를 빠뜨리지 않으면서도 이해하기 쉽게 한 문장으로 요약해 줘.”


## 3) 챗봇 만들기 실습 정답

In [7]:
class ChatSession:
    """Responses API 기반 멀티턴 챗봇 세션 관리 유틸리티 (정답)."""

    def __init__(
        self,
        client: OpenAI,
        model: str = "gpt-6-luna",
        developer_message: str = "You are a helpful assistant. You must answer in Korean and keep answers clear.",
        reasoning_effort: str = "medium",
        verbosity: str = "medium",
    ):
        self.client = client
        self.model = model
        self.developer_message = developer_message
        self.reasoning_effort = reasoning_effort
        self.verbosity = verbosity
        self.history = []
        self.last_response_id = None
        self.reset()

    def reset(self) -> None:
        """대화 히스토리를 초기 developer 메시지로 리셋합니다."""
        self.history = [
            {
                "role": "developer",
                "content": self.developer_message,
            }
        ]
        self.last_response_id = None

    def send(self, user_message: str):
        """사용자 메시지를 추가하고 모델 응답 텍스트와 response 객체를 반환합니다."""
        cleaned = user_message.strip()
        if not cleaned:
            raise ValueError("user_message는 비어 있을 수 없습니다.")

        self.history.append({"role": "user", "content": cleaned})

        message_text, response = request_chat_completion(
            client=self.client,
            model=self.model,
            history=self.history,
            reasoning_effort=self.reasoning_effort,
            verbosity=self.verbosity,
        )

        self.history.append({"role": "assistant", "content": message_text})
        self.last_response_id = response.id
        return message_text, response

    def pretty_history(self):
        """현재 히스토리를 읽기 쉬운 문자열로 변환합니다."""
        lines = []
        for msg in self.history:
            prefix = "사용자" if msg["role"] == "user" else "assistant" if msg["role"] == "assistant" else msg["role"]
            lines.append(f"[{prefix}] {msg['content']}")
        return "\n".join(lines)

In [8]:
chatbot = ChatSession(
    client,
    model="gpt-6-luna",
    developer_message="You are a helpful teaching assistant. Reply in Korean with actionable suggestions.",
    reasoning_effort="low",
    verbosity="medium",
)

In [9]:
reply, raw_response = chatbot.send("OpenAI API 실습을 복습하는 가장 좋은 방법을 알려 줘.")
print("모델 응답:\n", reply)

print("\n현재 히스토리:\n", chatbot.pretty_history())

모델 응답:
 OpenAI API 실습은 **다시 읽기보다, 기억해서 만들고 확인하는 방식**으로 복습하는 게 효과적입니다.

1. **핵심 흐름을 빈 화면에서 재현하기**  
   API 키 설정 → 요청 보내기 → 응답 확인하기를 예제 없이 한 번 구현해 보세요. 막힌 부분을 표시한 뒤 문서나 기존 코드를 확인합니다.

2. **작은 과제를 직접 만들기**  
   예를 들어 텍스트 요약기, 질문 답변기, JSON 형식으로 결과를 내는 도구처럼 실습 내용을 활용한 미니 프로젝트를 만들어 보세요. 입력을 바꾸며 응답도 시험해 봅니다.

3. **오류 상황도 연습하기**  
   키가 없거나 잘못된 경우, 요청 형식이 틀린 경우, 응답이 예상과 다른 경우를 확인해 보세요. 오류 메시지를 읽고 원인과 해결법을 짧게 기록하면 다음에 훨씬 빠르게 대응할 수 있습니다.

4. **응답을 검증하기**  
   결과가 원하는 형식인지, 누락되거나 부정확한 내용은 없는지 확인하세요. 필요하면 프롬프트를 고쳐 다시 실행하고, 무엇을 바꿨을 때 결과가 달라졌는지 비교합니다.

5. **복습 노트를 한 장으로 정리하기**  
   자주 쓰는 코드 흐름, 주요 설정, 오류 해결법, 보안 주의사항만 요약해 두세요. API 키는 코드에 직접 적거나 공개 저장소에 올리지 말고 환경 변수로 관리합니다.

**추천 복습 순서:** 기억만으로 재구현 → 미니 프로젝트 → 오류 시험 → 결과 점검 → 노트 정리. 다음 날 같은 과제를 짧게 다시 해 보면 기억에 오래 남습니다.

현재 히스토리:
 [developer] You are a helpful teaching assistant. Reply in Korean with actionable suggestions.
[사용자] OpenAI API 실습을 복습하는 가장 좋은 방법을 알려 줘.
[assistant] OpenAI API 실습은 **다시 읽기보다, 기억해서 만들고 확인하는 방식**으로 복습하는 게 효과적입니다.

1. **핵심 흐름을 빈

In [10]:
# 토큰 사용량도 한 번 확인해 보기
raw_response.usage

ResponseUsage(input_tokens=41, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=448, output_tokens_details=OutputTokensDetails(reasoning_tokens=41), total_tokens=489)

## 4) 위젯 기반 챗봇 UI (정답)
- (GitHub 미리보기에서는 위젯이 보이지 않아요. Colab이나 Jupyter에서 직접 실행해 보세요!)

In [11]:
import ipywidgets as widgets
from IPython.display import Markdown, display

chat_ui_session = ChatSession(
    client,
    model="gpt-6-luna",
    developer_message="You are a supportive study buddy. Reply in Korean with friendly tips.",
    reasoning_effort="medium",
)

log_output = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="8px"))
input_box = widgets.Text(
    placeholder="질문을 입력하고 Enter 또는 '전송'을 클릭하세요.",
    description="질문",
    layout=widgets.Layout(width="100%"),
)
send_button = widgets.Button(description="전송", button_style="primary")
reset_button = widgets.Button(description="초기화")
status_label = widgets.HTML(value="")


def append_message(role: str, text: str) -> None:
    with log_output:
        display(Markdown(f"**{role}**: {text}"))


def handle_send(_=None) -> None:
    text = input_box.value.strip()
    if not text:
        status_label.value = "<span style='color:#d9534f;'>메시지를 입력하세요.</span>"
        return

    status_label.value = ""
    append_message("사용자", text)
    input_box.value = ""
    send_button.disabled = True
    try:
        answer, _ = chat_ui_session.send(text)
        append_message("어시스턴트", answer)
    except Exception as exc:  # pragma: no cover
        append_message("오류", f"{type(exc).__name__}: {exc}")
    finally:
        send_button.disabled = False


def handle_reset(_=None) -> None:
    chat_ui_session.reset()
    log_output.clear_output()
    status_label.value = "<span style='color:#5cb85c;'>세션이 초기화되었습니다.</span>"


send_button.on_click(handle_send)
reset_button.on_click(handle_reset)

input_box.continuous_update = False
input_box.observe(lambda change: handle_send() if change['new'].strip() else None, names='value')

controls = widgets.HBox([send_button, reset_button, status_label])
ui = widgets.VBox([input_box, controls, log_output])

display(ui)

### 참고 자료
- https://platform.openai.com/docs
- https://platform.openai.com/docs/api-reference